In [3]:
import os
import time
import pandas as pd

This script is to identify which binding site on CpCBM92A the ligands B, C, and D are docked to, and create a table with CNN affinity and CNNscore per binding site (Trp) for the 20 runs of Gnina. 

In [4]:
PYMOL_PATH = 'C:\\Users\\susanntr\\AppData\Local\\pymol\\PyMOLWin.exe' 

#Pymol script (saved in command.pml file)
command = '''load PDBFILE
select site, chain CHAIN around 4.0
save site.pdb, site
quit
'''

In [5]:
def get_ligand_sites(filepath):
    idx = file.split('_')[-1].split('.')[0]   # Get out the index of the run, from the file name (eg. result_lig_B_0.log will give 0)
    for lig in ['B', 'C', 'D']:                               
        logfile = target_dir + f'result_lig_{lig}_{idx}.log'  # The logfile name from Gnina
        CNNscore, CNNaffinity = parse_log(logfile)            # Get out the CNN score and CNN affinity from the log file
         
        this_command = command.replace('PDBFILE', filepath)   # In the PyMOL command, insert the correct filepath for the pdb file of that ligand and index.
        this_command = this_command.replace('CHAIN', lig)     # Insert the ligand so that pymol selects the right chain. 
        with open('command.pml', 'w') as f:                   # We needed to store the command in the file for it to be opened by os.system
            f.write(this_command)
        
        os.system(f'{PYMOL_PATH} -q "command.pml"')           # produces site.pdb
        time.sleep(2)                                         # waits one second because pymol is slower than jupyter
        with open('site.pdb', 'r') as f:
            lines = f.readlines()
        os.remove('site.pdb')
        for line in lines:
            if 'TRP' in line:
                trp_number = line.split()[5]                  # Pymol will have selected residues close to the ligand. Now find the TRP res number in col 5.
        print(f'lig {lig}: W{trp_number} -- CNNscore:{CNNscore} -- CNNaffinity:{CNNaffinity}')

        for value in CNNscore:
            data[f'score_{trp_number}'].append(value)
        for value in CNNaffinity:
            data[f'affinity_{trp_number}'].append(value)
      

def parse_log(logfile):                                       # Reads out CNN affinities and CNN scores from the logfile and returns as lists
    cnn_scores = []
    cnn_affinities = []
    
    with open(logfile, 'r') as f:   #open and read logfile
        lines = f.readlines()
    
    
    for line in lines:              
        stripped = line.strip()
    
        if stripped and stripped[0].isdigit():   #If the line starts with a digit, it is a row in the table
            parts = stripped.split()
            cnn_scores.append(float(parts[3]))
            cnn_affinities.append(float(parts[4]))
    
    return cnn_scores, cnn_affinities

In [7]:
# Define the directory for a ligand and get out the affinities/CNNscores. 

lig = 'aCellobiose'

target_dir = f'OUTPUTS/new_gnina_results_{lig}/'  # make dictionary to store CNN scores and CNN affinities.
data = {                      
    'score_48': [],
    'score_90': [],
    'score_132': [],
    'affinity_48': [],
    'affinity_90': [],
    'affinity_132': [],
}

for root, dirs, files in os.walk(target_dir):   # Search files in the target directory
    for file in files:
        if file.startswith(f'{lig}_model'):      # Select the pdb files from boltz (using startswith because we might have several models from Boltz, in that case it will do the above several times)
            filepath = target_dir + file
            print(filepath)
            get_ligand_sites(filepath)          # Get out the sites and make the dictionary

OUTPUTS/new_gnina_results_aCellobiose/aCellobiose_model_0.pdb
lig B: W48 -- CNNscore:[0.8775, 0.8612, 0.8133, 0.7955, 0.7257, 0.6575, 0.6178, 0.6094, 0.5552, 0.5339, 0.5231, 0.5187, 0.5029, 0.4786, 0.4643, 0.4534, 0.4298, 0.4226, 0.3976, 0.3617] -- CNNaffinity:[3.109, 2.917, 3.034, 2.738, 2.96, 2.729, 2.88, 3.667, 3.637, 3.002, 2.86, 2.671, 2.504, 2.78, 2.944, 2.86, 2.859, 2.697, 2.561, 2.964]
lig C: W90 -- CNNscore:[0.9566, 0.956, 0.9513, 0.948, 0.9291, 0.9173, 0.9149, 0.9035, 0.8943, 0.8937, 0.8879, 0.8828, 0.8632, 0.8581, 0.8553, 0.8408, 0.8369, 0.8325, 0.8312, 0.82] -- CNNaffinity:[3.254, 2.957, 2.892, 2.905, 2.946, 2.843, 3.099, 3.102, 3.118, 2.837, 2.921, 3.078, 2.846, 2.867, 3.019, 2.903, 2.714, 2.786, 2.788, 2.798]
lig D: W132 -- CNNscore:[0.8882, 0.8499, 0.8161, 0.769, 0.7017, 0.6738, 0.6426, 0.6202, 0.567, 0.5572, 0.5549, 0.5476, 0.5095, 0.4917, 0.3477, 0.3437, 0.3378, 0.3361, 0.329, 0.3255] -- CNNaffinity:[2.915, 3.4, 2.859, 2.853, 3.585, 2.866, 2.773, 2.839, 3.069, 3.302, 3

In [8]:
# Make a dataframe with the values
df = pd.DataFrame(data)   
# df.to_csv(f'docking_data/{lig}.csv')  # Save the table

df

,score_48,score_90,score_132,affinity_48,affinity_90,affinity_132
0,0.8775,0.9566,0.8882,3.109,3.254,2.915
1,0.8612,0.9560,0.8499,2.917,2.957,3.400
2,0.8133,0.9513,0.8161,3.034,2.892,2.859
3,0.7955,0.9480,0.7690,2.738,2.905,2.853
4,0.7257,0.9291,0.7017,2.960,2.946,3.585
5,0.6575,0.9173,0.6738,2.729,2.843,2.866
6,0.6178,0.9149,0.6426,2.880,3.099,2.773
7,0.6094,0.9035,0.6202,3.667,3.102,2.839
8,0.5552,0.8943,0.5670,3.637,3.118,3.069
9,0.5339,0.8937,0.5572,3.002,2.837,3.302


In [9]:
df.describe()


,score_48,score_90,score_132,affinity_48,affinity_90,affinity_132
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.579985,0.888670,0.560460,2.918650,2.933650,2.971700
std,0.159211,0.045240,0.185197,0.294423,0.138433,0.382889
min,0.361700,0.820000,0.325500,2.504000,2.714000,2.325000
25%,0.461575,0.851675,0.346700,2.735750,2.841500,2.749500
50%,0.528500,0.890800,0.556050,2.870000,2.904000,2.862500
75%,0.674550,0.920250,0.680775,2.973500,3.033750,3.318500
max,0.877500,0.956600,0.888200,3.667000,3.254000,3.585000
